In [1]:
!pip install ogb

In [2]:
pip install torch_geometric

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import pandas as pd

# --- OGBマッピングの動的ロード (再現用) ---
# 実際のデータが読み込まれていることを確認します。
root_dir = './data'
mapping_path = os.path.join(root_dir, 'ogbn_arxiv/mapping/labelidx2arxivcategeory.csv.gz')

try:
    label_df = pd.read_csv(mapping_path)
    # IDをキー、カテゴリコードを値とする辞書を作成
    ARXIV_CATEGORY_NAMES = dict(zip(label_df['label idx'], label_df['arxiv category']))
except Exception:
    # 失敗した場合はダミーデータを使用（元のコードのロジックを再現）
    ARXIV_CATEGORY_NAMES = {i: f'Dummy_Cat_{i}' for i in range(40)}
# ----------------------------------------


# IDとカテゴリコードのペアを抽出し、データフレームを作成
category_data = []
# 0から39までのIDを明示的に処理
for cat_id in range(40):
    category_code = ARXIV_CATEGORY_NAMES.get(cat_id, 'N/A')
    category_data.append([cat_id, category_code])

# DataFrameの作成
df_categories = pd.DataFrame(category_data, columns=['ID', 'Arxiv Category Code'])

print("=== OGBN-Arxiv カテゴリ ID (0-39) 対照表 ===")
df_categories

=== OGBN-Arxiv カテゴリ ID (0-39) 対照表 ===


,ID,Arxiv Category Code
0,0,arxiv cs na
1,1,arxiv cs mm
2,2,arxiv cs lo
3,3,arxiv cs cy
4,4,arxiv cs cr
5,5,arxiv cs dc
6,6,arxiv cs hc
7,7,arxiv cs ce
8,8,arxiv cs ni
9,9,arxiv cs cc


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from torch_geometric.data import Data
from torch_geometric.utils import negative_sampling, to_dense_adj
from torchdiffeq import odeint_adjoint as odeint
from sklearn.metrics import roc_auc_score, average_precision_score
import umap
from scipy.interpolate import griddata
import networkx as nx
import random
import sys 
from scipy.stats import spearmanr

# --- グローバル設定 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

OUTPUT_DIR = "research_output_asd"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Output directory: {OUTPUT_DIR}")

sys.setrecursionlimit(3000)

# PyTorch 2.6 security patch
_original_load = torch.load
def unsafe_load(*args, **kwargs):
    kwargs.pop('weights_only', None) # Remove if present to avoid error
    return _original_load(*args, **kwargs, weights_only=False)
torch.load = unsafe_load

# --- OGB Loading Logic ---
try:
    from ogb.nodeproppred import PygNodePropPredDataset
    try:
        dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='./data')
        mapping_path = os.path.join(dataset.root, 'mapping', 'labelidx2arxivcategeory.csv.gz')
        label_df = pd.read_csv(mapping_path)
        ARXIV_CATEGORY_NAMES = dict(zip(label_df['label idx'], label_df['arxiv category']))
        print(f"✓ OGB-Arxiv Categories Loaded: {len(ARXIV_CATEGORY_NAMES)}")
    except Exception:
        print("Using Dummy Categories.")
        ARXIV_CATEGORY_NAMES = {i: f'Cat_{i}' for i in range(40)}
except ImportError:
    print("OGB not found. Using Mock Data.")
    ARXIV_CATEGORY_NAMES = {i: f'Cat_{i}' for i in range(40)}

# ==========================================
# 1. Data Pipeline
# ==========================================
class UniversalDataFactory:
    def __init__(self, dataset_name, root_dir='./data'):
        self.dataset_name = dataset_name
        self.root_dir = root_dir
        
    def load_data(self):
        print(f"Loading {self.dataset_name}...")
        try:
            from ogb.nodeproppred import PygNodePropPredDataset
            dataset = PygNodePropPredDataset(name='ogbn-arxiv', root=self.root_dir)
            data = dataset
        except Exception as e:
            print(f"Fallback to Synthetic Data: {e}")
            num_nodes = 2000
            edge_index = torch.randint(0, num_nodes, (2, 10000))
            x = torch.randn(num_nodes, 128)
            y = torch.randint(0, 40, (num_nodes,))
            node_year = torch.randint(2015, 2021, (num_nodes, 1))
            data = Data(x=x, edge_index=edge_index, y=y.unsqueeze(1), node_year=node_year)

        df_nodes = pd.DataFrame({
            'node_id': range(data.num_nodes),
            'year': data.node_year.numpy().flatten(),
            'category': data.y.numpy().flatten()
        })
        edge_index = data.edge_index.numpy()
        df_edges = pd.DataFrame({'source': edge_index, 'target': edge_index[1]})
        
        return {
            'df_nodes': df_nodes,
            'df_edges': df_edges,
            'node_features': data.x,
            'num_categories': len(np.unique(data.y.numpy()))
        }

class DynamicGraphBuilder:
    def __init__(self, data_dict):
        self.data = data_dict
        
    def build_snapshots(self):
        df_n, df_e = self.data['df_nodes'], self.data['df_edges']
        feats = self.data['node_features']
        years = sorted(df_n['year'].unique())
        snapshots = {}
        
        # OGB-Arxiv starts around ~1970 but densifies later. Focus on recent history.
        valid_years = [y for y in years if y >= 2014]
        
        print(f"Building snapshots for years: {valid_years}")
        for year in valid_years:
            # Cumulative Graph Construction (Standard for Citation Nets)
            mask_n = df_n['year'] <= year
            active_nodes = df_n[mask_n]['node_id'].values
            
            # Map old IDs to new contiguous range if needed, 
            # but OGB features are aligned by ID, so we keep original IDs 
            # and just mask operations.
            
            mask_e = (df_e['source'].isin(active_nodes)) & (df_e['target'].isin(active_nodes))
            current_edges = df_e[mask_e]
            
            edge_index = torch.tensor(current_edges[['source', 'target']].values.T, dtype=torch.long)
            
            # Label Y (category)
            y = torch.full((feats.shape,), -1, dtype=torch.long)
            y[active_nodes] = torch.tensor(df_n.loc[mask_n, 'category'].values, dtype=torch.long)
            
            snapshots[year] = Data(
                x=feats, 
                edge_index=edge_index, 
                num_nodes=feats.shape, # Maintain global node space
                y=y,
                active_mask=torch.tensor(mask_n.values) # Track who is active
            )
            
        return snapshots, self.data['num_categories'], feats.shape[1]

# ==========================================
# 2. PROPOSED MODEL: ASD-ODE
# (Attenuated Source Diffusion)
# ==========================================

class ASD_ODEFunc(nn.Module):
    """
    The core differential equation:
    dh/dt = [GNN(h) - h] + alpha * exp(-lambda * t) * Source(h0)
    
    Why it works:
    1. Diffusion [GNN(h)-h] propagates info.
    2. Source(h0) prevents oversmoothing (anchoring).
    3. Decay exp(-lambda*t) allows exploration early, consolidation later.
    """
    def __init__(self, gnn_layer, hidden_dim):
        super(ASD_ODEFunc, self).__init__()
        self.gnn = gnn_layer
        self.edge_index = None
        self.h0 = None
        
        # Transformation for the source signal
        self.source_encoder = nn.Linear(hidden_dim, hidden_dim)
        
        # Learnable Parameters for the Dynamics
        self.alpha = nn.Parameter(torch.tensor(1.0)) # Strength of source
        self.lambda_decay = nn.Parameter(torch.tensor(1.0)) # Rate of attenuation
        
        # Tracking for visualization
        self.history = {}

    def set_graph(self, edge_index, h0):
        self.edge_index = edge_index
        self.h0 = h0.detach() # Anchor to initial state

    def forward(self, t, h):
        # 1. Diffusion Term (The "Pull" of the graph)
        # GNN(h) is the aggregated neighborhood. 
        # GNN(h) - h approximates the Laplacian (Force towards mean).
        diffusion_term = self.gnn(h, self.edge_index) - h
        
        # 2. Attenuated Source Term (The "Anchor" to self)
        # alpha * e^(-lambda * t) * tanh(W * h0)
        decay_factor = torch.exp(-torch.abs(self.lambda_decay) * t)
        source_signal = torch.tanh(self.source_encoder(self.h0))
        source_term = self.alpha * decay_factor * source_signal
        
        # Total Derivative
        dh_dt = diffusion_term + source_term
        
        return dh_dt

class ASD_ODE(nn.Module):
    def __init__(self, in_dim, hidden_dim, use_gat=True):
        super(ASD_ODE, self).__init__()
        self.input_encoder = nn.Linear(in_dim, hidden_dim)
        
        if use_gat:
            # GAT head=2 for stability
            self.gnn_layer = GATConv(hidden_dim, hidden_dim // 2, heads=2, concat=True)
        else:
            self.gnn_layer = GCNConv(hidden_dim, hidden_dim)
            
        self.ode_func = ASD_ODEFunc(self.gnn_layer, hidden_dim)
        
        # Link Predictor
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def encode(self, x, edge_index, t_end=1.0):
        h0 = F.relu(self.input_encoder(x))
        
        # Setup ODE
        self.ode_func.set_graph(edge_index, h0)
        t_span = torch.tensor([0.0, t_end]).to(x.device)
        
        # Integrate
        # Using 'euler' or 'rk4' is faster, 'dopri5' is adaptive/precise
        trajectory = odeint(self.ode_func, h0, t_span, method='rk4', options={'step_size': 0.5})
        
        h_final = trajectory[-1]
        return h_final

    def predict_link(self, z, edge_index):
        src, dst = edge_index
        z_src = z[src]
        z_dst = z[dst]
        cat = torch.cat([z_src, z_dst], dim=-1)
        return self.decoder(cat).squeeze() # Logits

# ==========================================
# 3. BASELINES & ABLATIONS
# ==========================================

# 2nd Order ODE (Ablation: "Velocity Matters")
class Order2_ODEFunc(nn.Module):
    def __init__(self, gnn, hidden_dim):
        super(Order2_ODEFunc, self).__init__()
        self.gnn = gnn
        self.edge_index = None
        self.damping = nn.Parameter(torch.tensor(0.5))
        
    def set_graph(self, edge_index):
        self.edge_index = edge_index
        
    def forward(self, t, state):
        # State is [pos, vel] concatenated
        n = state.shape
        dim = state.shape[1] // 2
        h = state[:, :dim]
        v = state[:, dim:]
        
        # Spring force: GNN(h) - h
        spring = self.gnn(h, self.edge_index) - h
        # Damping force: -gamma * v
        friction = -torch.abs(self.damping) * v
        
        dh_dt = v
        dv_dt = spring + friction
        
        return torch.cat([dh_dt, dv_dt], dim=1)

class Order2_ODE(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super(Order2_ODE, self).__init__()
        self.enc = nn.Linear(in_dim, hidden_dim)
        self.gnn = GCNConv(hidden_dim, hidden_dim)
        self.func = Order2_ODEFunc(self.gnn, hidden_dim)
        self.dec = nn.Linear(hidden_dim*2, 1)
        self.hidden_dim = hidden_dim
        
    def encode(self, x, edge_index):
        h0 = F.relu(self.enc(x))
        v0 = torch.zeros_like(h0)
        state0 = torch.cat([h0, v0], dim=1)
        
        self.func.set_graph(edge_index)
        t_span = torch.tensor([0.0, 1.0]).to(x.device)
        traj = odeint(self.func, state0, t_span, method='rk4')
        
        # Return position only
        return traj[-1, :, :self.hidden_dim]
    
    def predict_link(self, z, edge_index):
        cat = torch.cat([z[edge_index], z[edge_index[1]]], dim=-1)
        return self.dec(cat).squeeze()

# Standard Static Baseline
class StaticGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super(StaticGAT, self).__init__()
        self.enc = nn.Linear(in_dim, hidden_dim)
        self.conv1 = GATConv(hidden_dim, hidden_dim, heads=2, concat=False)
        self.conv2 = GATConv(hidden_dim, hidden_dim, heads=2, concat=False)
        self.dec = nn.Linear(hidden_dim*2, 1)
        
    def encode(self, x, edge_index):
        x = F.relu(self.enc(x))
        x = F.elu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x
        
    def predict_link(self, z, edge_index):
        cat = torch.cat([z[edge_index], z[edge_index[1]]], dim=-1)
        return self.dec(cat).squeeze()

# ==========================================
# 4. Metrics & Evaluation (TGB Style)
# ==========================================

def calculate_dirichlet_energy(z, edge_index):
    """Quantifies oversmoothing. High energy = distinct features."""
    src, dst = edge_index
    # E = 1/N * sum_edges ||h_i - h_j||^2
    diff = z[src] - z[dst]
    energy = torch.norm(diff, dim=1).pow(2).mean()
    return energy.item()

def evaluate_mrr(model, z, pos_edges, num_nodes, num_neg=500):
    """
    Approximated MRR (Mean Reciprocal Rank).
    For each positive edge, sample K negatives.
    Rank pos edge among negatives.
    """
    model.eval()
    
    # Randomly sample a subset of positive edges for speed
    if pos_edges.shape[1] > 2000:
        perm = torch.randperm(pos_edges.shape[1])[:2000]
        eval_edges = pos_edges[:, perm]
    else:
        eval_edges = pos_edges

    pos_scores = model.predict_link(z, eval_edges)
    
    # Negative sampling: (num_eval_edges, num_neg)
    neg_src = eval_edges.repeat_interleave(num_neg)
    neg_dst = torch.randint(0, num_nodes, (neg_src.shape,), device=z.device)
    neg_edges = torch.stack([neg_src, neg_dst], dim=0)
    
    neg_scores = model.predict_link(z, neg_edges)
    neg_scores = neg_scores.view(-1, num_neg)
    
    # Calculate Ranks
    # pos_scores: [M], neg_scores: [M, K]
    # cat: [M, K+1] (pos is at index 0)
    all_scores = torch.cat([pos_scores.unsqueeze(1), neg_scores], dim=1)
    
    # Sort descending
    _, indices = torch.sort(all_scores, descending=True, dim=1)
    
    # Find where index 0 is (Rank)
    # indices == 0 gives boolean mask. nonzero gives coordinates.
    # we want the column index where value is 0.
    ranks = (indices == 0).nonzero(as_tuple=True)[1] + 1
    
    mrr = (1.0 / ranks.float()).mean().item()
    hits10 = (ranks <= 10).float().mean().item()
    
    return mrr, hits10

# ==========================================
# 5. Training Engine
# ==========================================

def train_and_eval(model_name, model, snapshots, years, epochs=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
    results = train_years = years[:-1] # Use up to second-to-last as train source
    
    print(f"\nTraining {model_name}...")
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        # Temporal Training Loop
        for t_idx in range(len(train_years)-1):
            curr_y = train_years[t_idx]
            next_y = train_years[t_idx+1]
            
            data_curr = snapshots[curr_y].to(device)
            data_next = snapshots[next_y].to(device)
            
            optimizer.zero_grad()
            
            # Forward (Encode current structure)
            z = model.encode(data_curr.x, data_curr.edge_index)
            
            # Predict Links in Next Snapshot (Inductive step)
            # Focus on NEW edges in next snapshot ideally, but simplified to all next edges
            pos_edges = data_next.edge_index
            
            # Negative Sampling
            neg_edges = negative_sampling(pos_edges, num_nodes=data_next.num_nodes)
            
            pos_scores = model.predict_link(z, pos_edges)
            neg_scores = model.predict_link(z, neg_edges)
            
            # BPR Loss (better for ranking than BCE)
            # loss = -log(sigmoid(pos - neg))
            # Align sizes randomly for BPR
            if pos_scores.shape!= neg_scores.shape:
                min_len = min(pos_scores.shape, neg_scores.shape)
                pos_scores = pos_scores[:min_len]
                neg_scores = neg_scores[:min_len]
                
            loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-15).mean()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # Evaluation on Last Year (Test)
        test_y = years[-1]
        prev_y = years[-2]
        data_test = snapshots[test_y].to(device)
        data_prev = snapshots[prev_y].to(device)
        
        with torch.no_grad():
            z_test = model.encode(data_prev.x, data_prev.edge_index)
            mrr, hits10 = evaluate_mrr(model, z_test, data_test.edge_index, data_test.num_nodes)
            energy = calculate_dirichlet_energy(z_test, data_test.edge_index)
            
        print(f"  Ep {epoch+1} | Loss: {total_loss:.4f} | MRR: {mrr:.4f} | Hits@10: {hits10:.4f} | Energy: {energy:.4f}")
        
        results.append({
            'Epoch': epoch+1,
            'MRR': mrr,
            'Energy': energy,
            'Model': model_name
        })
        
    return pd.DataFrame(results), model

# ==========================================
# 6. Visualization & Main
# ==========================================

def plot_energy_comparison(df_res):
    plt.figure(figsize=(10, 6))
    for model in df_res['Model'].unique():
        subset = df_res[df_res['Model'] == model]
        plt.plot(subset['Epoch'], subset['Energy'], marker='o', label=model)
    plt.xlabel('Training Epoch')
    plt.ylabel('Dirichlet Energy (Log Scale)')
    plt.yscale('log')
    plt.title('Oversmoothing Analysis: Dirichlet Energy Evolution')
    plt.legend()
    plt.grid(True, which="both", ls="--", alpha=0.3)
    plt.savefig(f"{OUTPUT_DIR}/dirichlet_energy.png")
    print("✓ Saved Energy Plot")

def main():
    # 1. Load & Build
    factory = UniversalDataFactory('ogbn-arxiv')
    raw = factory.load_data()
    builder = DynamicGraphBuilder(raw)
    snapshots, num_cats, feat_dim = builder.build_snapshots()
    years = sorted(snapshots.keys())
    
    # 2. Initialize Models
    HIDDEN = 64
    models = {
        'Static GAT': StaticGAT(feat_dim, HIDDEN).to(device),
        'Order2 ODE (InnoVelo-Old)': Order2_ODE(feat_dim, HIDDEN).to(device),
        'ASD-ODE (Proposed)': ASD_ODE(feat_dim, HIDDEN, use_gat=True).to(device)
    }
    
    # 3. Train & Compare
    all_res = []
    
    for name, model in models.items():
        df, trained_model = train_and_eval(name, model, snapshots, years, epochs=5)
        all_res.append(df)
        
        # Save model checkpoint
        torch.save(trained_model.state_dict(), f"{OUTPUT_DIR}/{name.split()}_model.pt")

    # 4. Final Analysis
    full_df = pd.concat(all_res)
    full_df.to_csv(f"{OUTPUT_DIR}/final_metrics.csv", index=False)
    
    print("\n=== FINAL RESULTS (Max MRR) ===")
    print(full_df.groupby('Model').max())
    
    plot_energy_comparison(full_df)
    
    # 5. Scientific Summary Generation
    best_model = full_df.loc.idxmax()]['Model']
    print(f"\n>>> CONCLUSION: The {best_model} achieved the best performance.")
    if 'ASD' in best_model:
        print("This supports the hypothesis that Attenuated Source terms are more effective")
        print("than Velocity/Momentum for citation network evolution.")

if __name__ == "__main__":
    main()

SyntaxError: invalid syntax (1327461869.py, line 465)